# Data Pre-processing

In [1]:
import sklearn as sk
import pandas as pd

## Import Data

In [2]:
df = pd.read_csv("US_Accidents_March23.csv")
len(df)

7728394

## Basic Data Preprocessing
To simplify your project a bit, use only the data instances that contain at most 1 missing value. According to our counts:
There are 3,554,549 data instances with no missing values:
There are 244,702 data instances with exactly one missing value.
Hence, your dataset for the project should contain 3,554,549 + 244,702 = 3,799,251

In [3]:
df.dropna(thresh=len(df.columns) - 1, inplace=True)
len(df)

3799251

In [4]:
df.drop(columns=['ID', 'Description', 'Country', 'Source', 'Turning_Loop'], inplace = True)
print(df.columns)

Index(['Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng',
       'End_Lat', 'End_Lng', 'Distance(mi)', 'Street', 'City', 'County',
       'State', 'Zipcode', 'Timezone', 'Airport_Code', 'Weather_Timestamp',
       'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
       'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)',
       'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing',
       'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station',
       'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Sunrise_Sunset',
       'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight'],
      dtype='str')


In [5]:
basic_df = df.copy(deep = True)

# Handle datetime -> integer by extracting integer values
basic_df["Start_Time"] = pd.to_datetime(basic_df["Start_Time"], format="ISO8601")
basic_df["End_Time"] = pd.to_datetime(basic_df["End_Time"], format="ISO8601")

basic_df["Duration_Minutes"] = (
    basic_df["End_Time"] - basic_df["Start_Time"]
).dt.total_seconds() / 60.0
basic_df["Start_Hour"] = basic_df["Start_Time"].dt.hour
basic_df["Start_DayOfWeek"] = basic_df["Start_Time"].dt.dayofweek  # 0 = Monday, 6 = Sunday
basic_df["Start_Month"] = basic_df["Start_Time"].dt.month
basic_df.drop(columns=["Start_Time", "End_Time"], inplace=True)

In [6]:
# Drop location cols other than lat/long
basic_df.drop(
    columns=[
        "City", "County", "Zipcode", "Airport_Code", "State"
    ],
    inplace=True,
)

In [7]:
# #Convert sunset measures
# sunset_fields = ["Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
# for f in sunset_fields:
#     basic_df[f] = basic_df[f].map({"Day": 1, "Night": 0}) # day = 1, night = 0

In [8]:
# # One-hot encode wind direction and weather condition as dense columns
# weather_columns = ["Wind_Direction", "Weather_Condition"]
# basic_df = pd.get_dummies(
#     basic_df,
#     columns=weather_columns,
#     dummy_na=True,
# )

In [9]:
#Drop timezone, weather
basic_df.drop(columns = ["Timezone", "Weather_Timestamp"], inplace = True)

In [11]:
# Fill missing numeric values one column at a time to avoid allocating another full-size frame.
for column in basic_df.columns:
    if (column == "Street" or column == "Weather_Condition" or column == "Wind_Direction"):
        continue
    
    missing = basic_df[column].isna()
    if missing.any():
        fill_value = basic_df[column].median()
        if pd.isna(fill_value):
            fill_value = 0
        basic_df[column] = basic_df[column].fillna(fill_value)

In [12]:
basic_df.info()

<class 'pandas.DataFrame'>
Index: 3799251 entries, 3402762 to 7728393
Data columns (total 36 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Severity               int64  
 1   Start_Lat              float64
 2   Start_Lng              float64
 3   End_Lat                float64
 4   End_Lng                float64
 5   Distance(mi)           float64
 6   Street                 str    
 7   Temperature(F)         float64
 8   Wind_Chill(F)          float64
 9   Humidity(%)            float64
 10  Pressure(in)           float64
 11  Visibility(mi)         float64
 12  Wind_Direction         str    
 13  Wind_Speed(mph)        float64
 14  Precipitation(in)      float64
 15  Weather_Condition      str    
 16  Amenity                bool   
 17  Bump                   bool   
 18  Crossing               bool   
 19  Give_Way               bool   
 20  Junction               bool   
 21  No_Exit                bool   
 22  Railway                bool 

In [13]:
basic_df.head()

,Severity,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Street,Temperature(F),Wind_Chill(F),Humidity(%),...,Traffic_Calming,Traffic_Signal,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight,Duration_Minutes,Start_Hour,Start_DayOfWeek,Start_Month
3402762,3,40.108910,-83.092860,40.112060,-83.031870,3.230,Outerbelt E,42.1,36.1,58.0,...,False,False,Night,Night,Night,Night,360.0,0,0,2
3402767,3,39.172393,-84.492792,39.170476,-84.501798,0.500,I-75 S,37.0,29.8,93.0,...,False,False,Day,Day,Day,Day,360.0,7,0,2
3402768,2,39.063240,-84.032430,39.067310,-84.058510,1.427,State Route 32,35.6,29.2,100.0,...,False,True,Day,Day,Day,Day,360.0,8,0,2
3402769,2,39.067080,-84.058550,39.063020,-84.032540,1.423,State Route 32,35.6,29.2,100.0,...,False,False,Day,Day,Day,Day,360.0,8,0,2
3402771,2,41.375310,-81.820170,41.367860,-81.821740,0.521,I-71 S,33.1,30.0,92.0,...,False,False,Day,Day,Day,Day,360.0,11,0,2


## Advanced Preprocessing 

TODO

# Export

In [14]:
%pip install pyarrow

basic_df.to_parquet(
    "basic_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\roast\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
